In [39]:
from glob import glob
import re
import pandas as pd

In [40]:

def load_plans(plans_dir: str) -> dict:
    plans = {}
    folder_pattern = re.compile(
        r'/(?P<system>[a-z]+)_(?P<nodes>\d+)/'
    )
    file_pattern = re.compile(
        r'/(?P<query>q\d+)_(?P<plan_type>[a-z]+)\.txt'
    )

    for p in glob(plans_dir):
        dir_match = re.search(folder_pattern, p)
        file_match = re.search(file_pattern, p)
        if not dir_match or not file_match:
            print(f"No match for {p}")
            break

        system = dir_match.group('system')
        node_count = int(dir_match.group('nodes'))
        query = file_match.group('query')
        plan_type = file_match.group('plan_type')

        key = (system, node_count, query, plan_type)
        with open(p, 'r') as f:
            content = f.read()
            if "ERROR" in content or content == "":
                print(f"ERROR in {p}")
                continue
            plans[key] = content

    return plans


def plan_coverage(plans: dict) -> pd.DataFrame:
    rows = [{
        'system': k[0],
        'node_count': k[1],
        'query': k[2],
        'plan_type': k[3]
    } for k in plans]

    return pd.DataFrame(rows)

In [41]:
plans = load_plans("../**/plans/*.txt")

PREFERRED = {
    'crdb':  'distsql',
    'tidb':  'analyze',
    'citus': 'analyze',
}

coverage = plan_coverage(plans)
coverage.pivot_table(
    index=['system', 'node_count'],
    columns='plan_type',
    values='query',
    aggfunc='count'
).head()


ERROR in ../citus_20/plans/q22_analyze.txt
ERROR in ../citus_20/plans/q22_verbose.txt
ERROR in ../citus_20/plans/q15_analyze.txt
ERROR in ../citus_20/plans/q20_analyze.txt
ERROR in ../citus_20/plans/q20_verbose.txt
ERROR in ../citus_20/plans/q17_verbose.txt
ERROR in ../citus_20/plans/q17_analyze.txt
ERROR in ../citus_10/plans/q22_analyze.txt
ERROR in ../citus_10/plans/q22_verbose.txt
ERROR in ../citus_10/plans/q15_analyze.txt
ERROR in ../citus_10/plans/q20_analyze.txt
ERROR in ../citus_10/plans/q20_verbose.txt
ERROR in ../citus_10/plans/q17_verbose.txt
ERROR in ../citus_10/plans/q17_analyze.txt
ERROR in ../tidb_3/plans/q22_analyze.txt
ERROR in ../tidb_3/plans/q22_verbose.txt
ERROR in ../tidb_3/plans/q5_analyze.txt
ERROR in ../tidb_3/plans/q21_analyze.txt
ERROR in ../tidb_3/plans/q18_analyze.txt
ERROR in ../tidb_3/plans/q20_analyze.txt
ERROR in ../tidb_5/plans/q5_analyze.txt
ERROR in ../tidb_5/plans/q21_analyze.txt
ERROR in ../tidb_5/plans/q7_analyze.txt
ERROR in ../tidb_5/plans/q18_ana

plan_type          analyze  distsql  verbose
system node_count                           
citus  3              18.0      NaN     19.0
       5              18.0      NaN     19.0
       10             18.0      NaN     19.0
       20             18.0      NaN     19.0
crdb   3               NaN     22.0     22.0